# Script to examine off-axis structure and direction

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
V_PERP_ROOT      = "/path/to/off_axis_output"  # OUTPUT_DIR from off_axis_save_raw
OUTPUT_DIR       = "/path/to/outputs/off_axis" # where analysis outputs are saved
LABELS_CSV       = "/path/to/partial_mask_geometry_xai/models_and_data/class_labels_indices.csv"
MIN_N_DATAPOINTS = 2000  # min vectors per retention bin; reduce for smaller datasets
# ─────────────────────────────────────────────────────────────────────────────
print(f"V_perp root : {V_PERP_ROOT}\n"
      f"Output dir  : {OUTPUT_DIR}\n"
      f"Labels CSV  : {LABELS_CSV}\n"
      f"Min n/bin   : {MIN_N_DATAPOINTS}")


In [15]:
import os
import numpy as np
import pandas as pd

_V_PERP = V_PERP_ROOT
_ASSETS = OUTPUT_DIR
_DIR = OUTPUT_DIR

os.makedirs(_DIR, exist_ok=True)

MODELS  = ["panns_no_specaug", "panns_specaug_trained", "ast_wrapper"]
FILLS   = ["zero", "mean", "gaussian_noise"]
CLASSES = sorted(["bagpipes", "boing", "chicken_rooster", "didgeridoo", "dog", "drum_kit",
                   "frog", "frying_food", "gunshot_gunfire", "hair_dryer", "harmonica",
                   "heart_sounds_heartbeat", "insect", "owl", "rain", "rub", "sewing_machine",
                   "speech", "thunder", "timpani", "train", "whispering"])

N_BINS           = 20
BIN_EDGES        = np.linspace(0.0, 1.0, N_BINS + 1)
BIN_CENTRES      = (BIN_EDGES[:-1] + BIN_EDGES[1:]) / 2

print("Init complete.")

Init complete.


In [16]:
def v_perp_covariance(model, fill):
    """
    Computes per-bin covariance of v_perp pooled across all classes for model x fill.
    Batch Welford update — stable against rounding errors (Schubert & Gertz, SSDBM 2018, Eq. 21).
    Returns (cov, n) where:
        cov : (N_BINS, 527, 527) float64 — Sigma_b for each bin
        n : (N_BINS,) int64 — number of vectors per bin
    """
    M2   = np.zeros((N_BINS, 527, 527), dtype=np.float64)
    mean = np.zeros((N_BINS, 527),      dtype=np.float64)
    n    = np.zeros(N_BINS,             dtype=np.int64)

    for c in CLASSES:
        clip_dir = os.path.join(_V_PERP, model, fill, c)
        if not os.path.isdir(clip_dir):
            continue
        for fname in os.listdir(clip_dir):
            if not fname.endswith(".npz"):
                continue
            with np.load(os.path.join(clip_dir, fname)) as npz:
                v_perp = npz["v_perp"].astype(np.float64)  # (N_masks, 527)
                ret    = npz["ret"].astype(np.float64)       # (N_masks,)

            valid  = (ret >= 0.0) & (ret <= 1.0)
            v_perp = v_perp[valid]
            ret    = ret[valid]
            b_idx  = np.digitize(ret, BIN_EDGES[1:-1])

            order  = np.argsort(b_idx)
            b_sort = b_idx[order]
            v_sort = v_perp[order]
            splits = np.searchsorted(b_sort, np.arange(N_BINS + 1))

            for b in range(N_BINS):
                vb = v_sort[splits[b]:splits[b + 1]]
                if len(vb) == 0:
                    continue
                k      = len(vb)
                mean_b = vb.mean(axis=0)       # (527,)
                delta  = mean_b - mean[b]      # (527,)
                n_new  = n[b] + k

                M2[b]  += (vb - mean_b).T @ (vb - mean_b) + np.outer(delta, delta) * (n[b] * k / n_new)
                mean[b] = mean[b] + k / n_new * delta
                n[b]    = n_new

    cov = np.full((N_BINS, 527, 527), np.nan)
    for b in range(N_BINS):
        if n[b] < MIN_N_DATAPOINTS:
            continue
        cov[b] = M2[b] / n[b]

    return cov, n

print("v_perp_covariance defined.")

v_perp_covariance defined.


# Sweep

In [ ]:
for m in MODELS:
    for f in FILLS:
        out_path = os.path.join(_DIR, f"{m}_{f}.npz")
        if os.path.isfile(out_path):
            print(f"Skipping {m} x {f} — already exists.")
            continue
        print(f"Computing {m} x {f}...")
        cov, n = v_perp_covariance(m, f)
        np.savez(out_path, cov=cov, n=n)
        print(f"  Saved to {out_path}.")

print("Sweep complete.")

In [21]:
DIM = 527

data = {}
for m in MODELS:
    for f in FILLS:
        with np.load(os.path.join(_DIR, f"{m}_{f}.npz")) as loaded:
            data[(m, f)] = {"cov": loaded["cov"].copy(), "n": loaded["n"].copy()}

rows = []
for m, f in data:
    for b in range(N_BINS):
        nb = data[(m, f)]["n"][b]
        if nb < MIN_N_DATAPOINTS:
            continue
        cov_b   = data[(m, f)]["cov"][b]            # (527, 527)
        eigvals = np.linalg.eigvalsh(cov_b)          # (527,) ascending

        min_eig = float(eigvals.min())
        n_ok    = nb >= DIM

        # participation ratio — clamp tiny negatives from float64 rounding;
        # without clamping, rounding artefacts inflate the denominator and deflate PR
        ev = np.maximum(eigvals, 0)
        pr = ev.sum()**2 / (ev**2).sum()

        rows.append({
            "model":   m,
            "fill":    f,
            "bin":     b,
            "alpha":   round(float(BIN_CENTRES[b]), 3),
            "n":       int(nb),
            "n_ok":    n_ok,
            "min_eig": round(min_eig, 6),
            "pr":      round(float(pr), 2),
        })

df_eig = pd.DataFrame(rows)

print("=== Check 1: negative eigenvalues ===")
bad = df_eig[df_eig["min_eig"] < -1e-6]
print(f"  {len(bad)} bins with min_eig < -1e-6" if len(bad) else "  None — all clean.")

print("\n=== Check 2: n vs dimensionality (need n >= 527) ===")
low_n = df_eig[~df_eig["n_ok"]]
print(f"  {len(low_n)} bins with n < 527" if len(low_n) else "  None — all bins well-powered.")

print("\n=== Participation ratio (mean across bins) ===")
print(df_eig.groupby(["model", "fill"])[["pr"]].mean().round(2))

=== Check 1: negative eigenvalues ===
  None — all clean.

=== Check 2: n vs dimensionality (need n >= 527) ===
  None — all bins well-powered.

=== Participation ratio (mean across bins) ===
                                         pr
model                 fill                 
ast_wrapper           gaussian_noise  15.13
                      mean            13.37
                      zero            19.14
panns_no_specaug      gaussian_noise  16.03
                      mean            19.56
                      zero             8.39
panns_specaug_trained gaussian_noise  21.20
                      mean            16.75
                      zero            17.62


In [22]:
_LABELS_CSV = LABELS_CSV
labels = pd.read_csv(_LABELS_CSV).set_index("index")["display_name"].to_dict()

TOP_K = 10

print("=== Top classes by mean per-class variance in v_perp ===")
print("(diagonal of Sigma_b, averaged across valid bins)\n")

for m in MODELS:
    for f in FILLS:
        cov = data[(m, f)]["cov"]
        n   = data[(m, f)]["n"]

        diag_sum = np.zeros(DIM)
        n_valid  = 0
        for b in range(N_BINS):
            if n[b] < MIN_N_DATAPOINTS:
                continue
            diag_sum += np.diag(cov[b])
            n_valid  += 1

        mean_var = diag_sum / max(n_valid, 1)
        top_idx  = np.argsort(mean_var)[::-1][:TOP_K]
        top      = [(labels[int(i)], round(float(mean_var[i]), 4)) for i in top_idx]

        print(f"[{m}  x  {f}]")
        for name, var in top:
            print(f"  {var:.4f}  {name}")
        print()

print("=== Leading eigenvector class loadings (bin-averaged) ===")
print("(mean |loading| on the top eigenvector across valid bins)\n")

for m in MODELS:
    for f in FILLS:
        cov = data[(m, f)]["cov"]
        n   = data[(m, f)]["n"]

        loading_sum = np.zeros(DIM)
        n_valid = 0
        for b in range(N_BINS):
            if n[b] < MIN_N_DATAPOINTS:
                continue
            _, eigvecs = np.linalg.eigh(cov[b])     # ascending order
            loading_sum += np.abs(eigvecs[:, -1])    # last col = leading eigvec
            n_valid += 1

        if n_valid == 0:
            continue
        mean_loading = loading_sum / n_valid
        top_idx = np.argsort(mean_loading)[::-1][:TOP_K]
        top = [(labels[int(i)], round(float(mean_loading[i]), 4)) for i in top_idx]

        print(f"[{m}  x  {f}]")
        for name, load in top:
            print(f"  {load:.4f}  {name}")
        print()

=== Top classes by mean per-class variance in v_perp ===
(diagonal of Sigma_b, averaged across valid bins)

[panns_no_specaug  x  zero]
  9.1844  Dial tone
  7.2888  Sidetone
  6.7867  Smoke detector, smoke alarm
  6.5836  Squeal
  6.1136  Busy signal
  5.8946  Whistle
  5.8518  Toothbrush
  5.6057  Boing
  5.5563  Buzzer
  5.3880  Whistling

[panns_no_specaug  x  mean]
  3.7655  Mains hum
  2.8567  Hum
  2.6531  Smoke detector, smoke alarm
  2.5939  Busy signal
  2.5912  Fire alarm
  2.4865  Buzzer
  2.3047  Sidetone
  2.2870  Fart
  2.1814  Toothbrush
  2.1614  Dial tone

[panns_no_specaug  x  gaussian_noise]
  4.4705  Speech
  3.7723  Cricket
  3.4912  Stomach rumble
  3.2021  Clicking
  3.1831  Owl
  3.0376  Chirp, tweet
  2.9718  Mains hum
  2.9090  Mouse
  2.8909  Environmental noise
  2.8039  Sonar

[panns_specaug_trained  x  zero]
  2.0604  Plop
  1.4633  Breaking
  1.3488  Boing
  1.2266  Shatter
  1.2224  Wail, moan
  1.1956  Sidetone
  1.1850  Toothbrush
  1.1819  Dial tone


In [23]:
TOP_K = 10

print("=== Leading-eigenvector stability across alpha ===\n")
print(f"{'Condition':<45} {'mean |cos|':>10}  {'min |cos|':>9}  {'ev frac%':>9}  {'top class (mean loading)'}")
print("-" * 115)

stability_rows = []

for m in MODELS:
    for f in FILLS:
        n   = data[(m, f)]["n"]
        cov = data[(m, f)]["cov"]

        valid = [b for b in range(N_BINS) if n[b] >= MIN_N_DATAPOINTS]
        if len(valid) < 2:
            print(f"{m} x {f}: skipped — fewer than 2 valid bins")
            continue

        # leading eigenvector and leading eigenvalue fraction at each valid bin
        evecs   = []
        ev_fracs = []
        for b in valid:
            eigvals, eigvecs = np.linalg.eigh(cov[b])    # ascending order
            evecs.append(eigvecs[:, -1])
            ev = np.maximum(eigvals, 0)
            ev_fracs.append(float(ev[-1] / ev.sum()))     # fraction of variance in leading eigvec
        evecs    = np.stack(evecs)                         # (n_valid, 527)
        mean_ev_frac = float(np.mean(ev_fracs))

        # pairwise absolute cosine similarity — |cos| handles sign ambiguity
        cos_mat  = np.abs(evecs @ evecs.T)                 # (n_valid, n_valid)
        upper    = cos_mat[np.triu_indices(len(valid), k=1)]
        mean_cos = float(upper.mean())
        min_cos  = float(upper.min())

        # mean absolute loading → identifies classes defining the stable direction
        mean_loading = np.abs(evecs).mean(axis=0)          # (527,)
        top_class    = labels[int(np.argmax(mean_loading))]

        label = f"{m} x {f}"
        print(f"{label:<45} {mean_cos:>10.4f}  {min_cos:>9.4f}  {mean_ev_frac*100:>8.1f}%  {top_class}")

        stability_rows.append({
            "model":       m,
            "fill":        f,
            "mean_cos":    round(mean_cos, 4),
            "min_cos":     round(min_cos, 4),
            "ev_frac_pct": round(mean_ev_frac * 100, 1),
            "top_class":   top_class,
        })

df_stability = pd.DataFrame(stability_rows)

print("\n=== Top classes on the mean leading eigenvector ===\n")
for row in stability_rows:
    m, f = row["model"], row["fill"]
    n_   = data[(m, f)]["n"]
    cov_ = data[(m, f)]["cov"]

    valid = [b for b in range(N_BINS) if n_[b] >= MIN_N_DATAPOINTS]
    evecs = np.stack([np.linalg.eigh(cov_[b])[1][:, -1] for b in valid])
    mean_loading = np.abs(evecs).mean(axis=0)
    top_idx = np.argsort(mean_loading)[::-1][:TOP_K]

    ev_frac_str = f"  [ev frac {row['ev_frac_pct']:.1f}%  mean|cos| {row['mean_cos']:.4f}]"
    print(f"[{m}  x  {f}]{ev_frac_str}")
    if row["mean_cos"] < 0.90:
        print("  *** direction unstable — loadings below are unreliable ***")
    for i in top_idx:
        print(f"  {mean_loading[i]:.4f}  {labels[int(i)]}")
    print()

=== Leading-eigenvector stability across alpha ===

Condition                                     mean |cos|  min |cos|   ev frac%  top class (mean loading)
-------------------------------------------------------------------------------------------------------------------
panns_no_specaug x zero                           0.9535     0.6843      32.1%  Squeal
panns_no_specaug x mean                           0.8583     0.2041      15.3%  Dental drill, dentist's drill
panns_no_specaug x gaussian_noise                 0.9046     0.5174      18.5%  Mechanical fan
panns_specaug_trained x zero                      0.7134     0.2404      21.6%  Plop
panns_specaug_trained x mean                      0.8246     0.2798      16.7%  Mains hum
panns_specaug_trained x gaussian_noise            0.6079     0.0200      13.1%  Mouse
ast_wrapper x zero                                0.9461     0.7439      18.5%  Snake
ast_wrapper x mean                                0.9372     0.7568      23.5%  Busy sig

In [24]:

# Export summary table for paper
_OUT_DIR = OUTPUT_DIR
os.makedirs(_OUT_DIR, exist_ok=True)

# Mean PR per condition (across valid bins)
pr_summary = (
    df_eig.groupby(["model", "fill"])
    .agg(
        pr_mean   = ("pr", "mean"),
        pr_min    = ("pr", "min"),
        pr_max    = ("pr", "max"),
        n_bins    = ("pr", "count"),
        n_median  = ("n", "median"),
        n_min     = ("n", "min"),
        n_max     = ("n", "max"),
    )
    .round(2)
    .reset_index()
)

# Join with directional stability metrics
summary = pr_summary.merge(df_stability, on=["model", "fill"])

out_path = os.path.join(_OUT_DIR, "off_axis_summary.csv")
summary.to_csv(out_path, index=False)
print(f"Saved → {out_path}\n")
print(summary[["model", "fill", "pr_mean", "mean_cos", "min_cos", "ev_frac_pct", "top_class"]].to_string(index=False))


Saved → /Users/nicolasgarcia/Documents/phd/phd-writing/icassp-2027/data/off_axis_summary.csv

                model           fill  pr_mean  mean_cos  min_cos  ev_frac_pct                     top_class
          ast_wrapper gaussian_noise    15.13    0.8699   0.2505         20.0                      Sidetone
          ast_wrapper           mean    13.37    0.9372   0.7568         23.5                   Busy signal
          ast_wrapper           zero    19.14    0.9461   0.7439         18.5                         Snake
     panns_no_specaug gaussian_noise    16.03    0.9046   0.5174         18.5                Mechanical fan
     panns_no_specaug           mean    19.56    0.8583   0.2041         15.3 Dental drill, dentist's drill
     panns_no_specaug           zero     8.39    0.9535   0.6843         32.1                        Squeal
panns_specaug_trained gaussian_noise    21.20    0.6079   0.0200         13.1                         Mouse
panns_specaug_trained           mean    16